<a href="https://colab.research.google.com/github/ryanatberkeley/aeesp-grand-challenges-nlp-pipeline/blob/main/bertopic_visualizations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# BERTopic + SPECTER2 visualizations

This notebook mirrors `bertopic_modeling.ipynb`: **SPECTER2** (`allenai/specter2_base`) embeddings and **guided (zero-shot) BERTopic** with the six NASEM Grand Challenge seed topics. It then generates the main **Plotly** figures BERTopic supports, plus **UMAP views** and **topic frequency** plots from the same embeddings and assignments.

**Input:** `works_df_*.csv` from `aseep_cleanedcorpus.ipynb` (set `CSV_GLOB_PATTERN` for Drive paths).

**Dependencies:** `bertopic`, `sentence-transformers`, `umap-learn`, `plotly` (optional: `kaleido` for static PNG export).

**Note:** APIs differ slightly across BERTopic versions—cells catch errors and print a skip message.

**Plots included:** (1) inter-topic map (2) term barchart (3) topic similarity heatmap (4) hierarchy (5) term rank (6) document map (7) one-doc probability distribution (8) topics over time (9) UMAP by topic (10) counts per topic (11) embedding cohesion diagnostic (12) HTML export (13) optional save.


## Install dependencies


In [ ]:
try:
    from bertopic import BERTopic
except ImportError:
    !pip install -q bertopic
    from bertopic import BERTopic

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    !pip install -q sentence_transformers
    from sentence_transformers import SentenceTransformer

try:
    import umap
except ImportError:
    !pip install -q umap-learn
    import umap

import glob
import os
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)


## Configuration and load `works_df_*.csv`


In [ ]:

MIN_PUBLICATION_YEAR = 2000
ABSTRACT_PLACEHOLDER = "No abstract available"
CSV_GLOB_PATTERN = "works_df_*.csv"
UMAP_RANDOM_STATE = 42
SPECTER_MODEL_NAME = "allenai/specter2_base"
ZEROSHOT_MIN_SIMILARITY = 0.5

grand_challenges = [
        "Energy efficient water conservation strategies and technologies: energy-efficient water systems, water conservation, rainwater harvesting, smart water systems, sustainable water management, water reuse, water recovery, stormwater reuse, graywater treatment, stakeholder input, behavioral science, social science, water demand management, leak detection, water loss control, smart metering, water pricing, drip irrigation, precision irrigation, urban water management, integrated urban water management, nonpotable reuse",
        "Low-cost desalination and water reuse technologies: water reuse, water reclamation, water recovery, wastewater recovery, reclaimed water use, desalination, brackish water treatment, membrane treatment, membrane separation, high-pressure membrane filtration, advanced membranes, polymers, reverse osmosis, nanofiltration, electrodialysis, brine management, concentrate minimization, zero liquid discharge, ZLD, solar desalination, solar distillation, forward osmosis, membrane distillation, capacitive deionization, ion exchange, advanced oxidation processes, UV advanced oxidation, hydroxyl radicals, AOPs, brine valorization, circular water systems, energy-water nexus",
        "Water supply and water quality forecasting tools: sensors, sensor networks, water reliability, water supply, water infrastructure, water security, smart phone, wireless, pathogens, water quality monitoring, biosensors, remote sensing, Internet of Things, smart water networks, early warning systems, contamination event detection, predictive modeling, machine learning, artificial intelligence, digital twin, anomaly detection, decision support systems, spatiotemporal analysis, hydrologic modeling, hydraulic modeling, drought forecasting",
        "Energy-neutral or energy-positive cost-effective wastewater treatment technologies: water recovery, water reuse, pathogens, low cost, microbial fuel cells, anaerobic digestion, biogas production, primary wastewater treatment, secondary wastewater treatment, nutrient recovery, phosphorus recovery, nitrogen recovery, biosolids, sludge treatment, anaerobic membrane bioreactor, circular sanitation, decentralized sanitation, onsite sanitation, nature-based treatment, constructed wetlands, lagoon systems, micropollutant removal, pharmaceutical removal, trace contaminant removal",
        "Water, sanitation, and hygiene challenges in low-income countries: user acceptance, stakeholders, no water toilets, onsite water treatment, decentralized water treatment, point of use water treatment, behavioral science, social science, policy, WASH, basic sanitation, sanitation access, hygiene access, fecal sludge management, septic systems, household water treatment, chlorination, solar disinfection, hand hygiene, hygiene behavior",
        "Aging water infrastructure: water infrastructure, water distribution systems, lead pipes, adaptive systems, aging infrastructure, asset management, pipe failure, water main breaks, corrosion control, copper corrosion, biofilm, nitrification, hydraulic transients, water age, leakage, service reliability, risk-based prioritization, infrastructure resilience, pipe replacement, smart infrastructure",
    ]


def filter_papers_for_topic_modeling(df, min_year, abstract_placeholder):
    out = df[df["publication_year"] >= min_year]
    out = out.dropna(subset=["title"])
    out = out[out["title"].astype(str).str.strip() != ""]
    out = out.dropna(subset=["abstract"])
    out = out[out["abstract"] != abstract_placeholder]
    out = out[out["abstract"].astype(str).str.strip() != ""]
    return out


file_paths = sorted(glob.glob(CSV_GLOB_PATTERN))
if not file_paths:
    raise FileNotFoundError(
        "No CSV files matched CSV_GLOB_PATTERN. Upload works_df_*.csv or set CSV_GLOB_PATTERN."
    )

dfs = [pd.read_csv(f) for f in file_paths]
df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df)} rows from {len(file_paths)} file(s).")

df = filter_papers_for_topic_modeling(df, MIN_PUBLICATION_YEAR, ABSTRACT_PLACEHOLDER)
print(f"After filters: {len(df)} papers.")

if "keywords" not in df.columns:
    df["keywords"] = ""

df["title"] = df["title"].fillna("")
df["keywords"] = df["keywords"].fillna("")
df["combined_text"] = df["title"] + " " + df["keywords"] + " " + df["abstract"]
docs = df["combined_text"].tolist()



## Encode documents with SPECTER2 and fit guided BERTopic


In [ ]:
print("Loading SPECTER2 and encoding documents...")
specter_model = SentenceTransformer(SPECTER_MODEL_NAME)
embeddings = specter_model.encode(docs, show_progress_bar=True, batch_size=32)

topic_model = BERTopic(
    embedding_model=specter_model,
    zeroshot_topic_list=grand_challenges,
    zeroshot_min_similarity=ZEROSHOT_MIN_SIMILARITY,
    language="english",
    calculate_probabilities=True,
    verbose=True,
)

try:
    topics, probabilities = topic_model.fit_transform(docs, embeddings=embeddings)
except TypeError:
    topics, probabilities = topic_model.fit_transform(docs, embeddings)

df["assigned_topic_id"] = topics
topic_info = topic_model.get_topic_info()
display(topic_info)
print("Unique topic ids (incl. -1 outliers):", sorted(set(topics)))


## UMAP: 2-D layout of SPECTER2 vectors (for `visualize_documents` + custom plots)


In [ ]:
n_samples = embeddings.shape[0]
n_neighbors = min(15, max(2, n_samples - 1))
reducer = umap.UMAP(
    n_neighbors=n_neighbors,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=UMAP_RANDOM_STATE,
)
reduced_embeddings = reducer.fit_transform(embeddings)
print("UMAP shape:", reduced_embeddings.shape)


### 1. Inter-topic distance map — `visualize_topics()`


In [ ]:
fig = topic_model.visualize_topics()
fig.show()


### 2. Top terms per topic — `visualize_barchart()`


In [ ]:
fig = topic_model.visualize_barchart(top_n_topics=min(12, len(topic_info)))
fig.show()


### 3. Pairwise topic similarity — `visualize_heatmap()`


In [ ]:
try:
    fig = topic_model.visualize_heatmap()
    fig.show()
except Exception as e:
    print("visualize_heatmap skipped:", e)


### 4. Topic hierarchy (dendrogram) — `visualize_hierarchy()`


In [ ]:
try:
    fig = topic_model.visualize_hierarchy()
    fig.show()
except Exception as e:
    print("visualize_hierarchy skipped:", e)


### 5. c-TF-IDF term ranks within one topic — `visualize_term_rank()`


In [ ]:
cids = [t for t in topic_info["Topic"].tolist() if t != -1]
topic_id = cids[0] if cids else None
if topic_id is not None:
    try:
        fig = topic_model.visualize_term_rank(topic=topic_id)
        fig.show()
    except Exception as e:
        print("visualize_term_rank skipped:", e)
else:
    print("No non-outlier topics to show.")


### 6. Document embedding map — `visualize_documents()` (uses SPECTER2 + UMAP)


In [ ]:
try:
    fig = topic_model.visualize_documents(
        docs,
        topics=topics,
        embeddings=embeddings,
        reduced_embeddings=reduced_embeddings,
        sample=0.35 if len(docs) > 800 else 1.0,
    )
    fig.show()
except TypeError:
    fig = topic_model.visualize_documents(docs, topics=topics, embeddings=embeddings)
    fig.show()
except Exception as e:
    print("visualize_documents skipped:", e)


### 7. Topic probability bar chart for one document — `visualize_distribution()`


In [ ]:
if probabilities is not None and len(probabilities) > 0:
    try:
        fig = topic_model.visualize_distribution(probabilities[0], min_probability=0.01)
        fig.show()
    except Exception as e:
        print("visualize_distribution skipped:", e)
else:
    print("No probability matrix returned.")


### 8. Topic evolution vs. time — `topics_over_time()` + `visualize_topics_over_time()`


In [ ]:
if "publication_year" in df.columns and df["publication_year"].notna().any():
    timestamps = df["publication_year"].astype(float).tolist()
    try:
        tot = topic_model.topics_over_time(docs, topics, timestamps)
        fig = topic_model.visualize_topics_over_time(tot)
        fig.show()
    except Exception as e:
        print("topics_over_time skipped:", e)
else:
    print("No publication_year column.")


### 9. Custom: UMAP scatter — embeddings colored by assigned topic


In [ ]:
tid = np.array(topics)
topic_label = np.where(tid == -1, "Outlier (-1)", "Topic " + tid.astype(str))
umap_df = pd.DataFrame({
    "UMAP-1": reduced_embeddings[:, 0],
    "UMAP-2": reduced_embeddings[:, 1],
    "topic": topic_label,
    "title": df["title"].astype(str).str.slice(0, 100),
})
fig = px.scatter(
    umap_df,
    x="UMAP-1",
    y="UMAP-2",
    color="topic",
    hover_data=["title"],
    title="SPECTER2 embeddings → UMAP, colored by BERTopic assignment",
    opacity=0.55,
)
fig.update_layout(width=1100, height=700, legend_itemsizing="constant")
fig.show()


### 10. Custom: papers per topic (bar chart)


In [ ]:
counts = pd.Series(topics).value_counts().sort_index()
fig = px.bar(x=counts.index.astype(str), y=counts.values, labels={"x": "Topic id", "y": "Count"}, title="Documents per topic")
fig.show()


### 11. Custom: mean SPECTER2 cosine similarity to topic centroid (diagnostic)


In [ ]:
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

E = normalize(embeddings, norm="l2", axis=1)
centroids = []
labels = []
for t in sorted(set(topics)):
    mask = np.array(topics) == t
    if mask.sum() == 0:
        continue
    c = E[mask].mean(axis=0, keepdims=True)
    c = normalize(c, norm="l2", axis=1).ravel()
    sims = cosine_similarity(E[mask], c.reshape(1, -1)).ravel()
    centroids.append({"topic": t, "mean_cosine_to_centroid": float(sims.mean())})
    labels.append(t)
diag = pd.DataFrame(centroids)
display(diag)
fig = px.bar(diag, x="topic", y="mean_cosine_to_centroid", title="Internal cohesion (higher = tighter cluster in embedding space)")
fig.show()


### 12. Export Plotly figures to HTML


In [ ]:
out_dir = "bertopic_figures"
os.makedirs(out_dir, exist_ok=True)

def _save(name, factory):
    try:
        fig = factory()
        path = os.path.join(out_dir, f"{name}.html")
        fig.write_html(path)
        print("Wrote", path)
    except Exception as e:
        print(name, "failed:", e)

def _umap_by_topic_fig():
    tid = np.array(topics)
    lab = np.where(tid == -1, "Outlier (-1)", "Topic " + tid.astype(str))
    udf = pd.DataFrame({
        "UMAP-1": reduced_embeddings[:, 0],
        "UMAP-2": reduced_embeddings[:, 1],
        "topic": lab,
        "title": df["title"].astype(str).str.slice(0, 100),
    })
    return px.scatter(udf, x="UMAP-1", y="UMAP-2", color="topic", hover_data=["title"])

_save("intertopic_distance", lambda: topic_model.visualize_topics())
_save("barchart_top_terms", lambda: topic_model.visualize_barchart(top_n_topics=min(12, len(topic_info))))
_save("umap_by_topic", _umap_by_topic_fig)
print("Optional: zip bertopic_figures/ and download from Colab.")


### 13. Optional: save model and arrays for offline plots


In [ ]:
# topic_model.save("bertopic_specter2")
# np.savez_compressed("specter2_run.npz", embeddings=embeddings, topics=np.array(topics), reduced=reduced_embeddings)
# print("Uncomment to persist model and arrays.")
